In [ ]:
from moviepy import * #moviePy cần binary ffpeg
import json
import gdown
from dataclasses import dataclass, field
from typing import List, Optional

In [5]:
@dataclass
class Animation:
    type: str
    startTime_sec: Optional[float] = None
    duration_sec: Optional[float] = None
    start_zoom: Optional[float] = None
    end_zoom: Optional[float] = None
    direction: Optional[str] = None

@dataclass
class Position:
    x: int
    y: int
    anchor: str

@dataclass
class Layer:
    layer_id: str
    type: str
    content: Optional[str] = None
    url: Optional[str] = None
    font: Optional[str] = None
    size: Optional[int] = None
    color: Optional[str] = None
    position: Optional[Position] = None
    animation: Optional[Animation] = None

@dataclass
class Scene:
    scene_id: int
    slide_id: int
    audioUrl: Optional[str]
    audioDuration_sec: float
    layers: List[Layer] = field(default_factory=list)

@dataclass
class VideoMetadata:
    title: str
    resolution: str
    fps: int

@dataclass
class VideoProject:
    metadata: VideoMetadata
    scenes: List[Scene]

# Hàm tạo object videoProject
def load_project_from_json(json_path: str) -> VideoProject:
    with open(json_path, "r") as f:
        data = json.load(f)

    metadata = VideoMetadata(
        title=data["videoMetadata"]["title"],
        resolution=data["videoMetadata"]["resolution"],
        fps=data["videoMetadata"]["fps"]
    )

    scenes = []
    for s in data["scenes"]:
        layers = []
        for l in s["layers"]:
            position = None
            if "position" in l:
                position = Position(**l["position"])
            animation = None
            if "animation" in l:
                animation = Animation(**l["animation"])
            layers.append(Layer(
                layer_id=l["layer_id"],
                type=l["type"],
                content=l.get("content"),
                url=l.get("url"),
                font=l.get("font"),
                size=l.get("size"),
                color=l.get("color"),
                position=position,
                animation=animation
            ))
        scenes.append(Scene(
            scene_id=s["scene_id"],
            slide_id=s["slide_id"],
            audioUrl=s.get("audioUrl"),
            audioDuration_sec=s["audioDuration_sec"],
            layers=layers
        ))

    return VideoProject(metadata=metadata, scenes=scenes)

# Main
project = load_project_from_json("scene_composition_agent_output.json")
print(project.metadata)
print(f"Loaded {len(project.scenes)} scenes")

# Thông tin từng scence
for scene in project.scenes:
    print(f"\nScene {scene.scene_id} (Slide {scene.slide_id})")
    print(f"Audio URL: {scene.audioUrl}")
    print(f"Audio Duration: {scene.audioDuration_sec} seconds")
    for layer in scene.layers:
        if layer.type == "text":
            print(f" [Text] {layer.layer_id}: {layer.content}")
        elif layer.type == "image":
            print(f" [Image] {layer.layer_id}: {layer.url}")
        elif layer.type == "color":
            print(f" [Color] {layer.layer_id}: {layer.color}")

VideoMetadata(title='Automate Your Content From Idea to LIVE Article in Minutes', resolution='1920x1080', fps=30)
Loaded 8 scenes

Scene 1 (Slide 1)
Audio URL: https://drive.google.com/file/d/16yvluYop5Q3rqb5EXa8rK_1BnsE_cMFO/view?usp=drivesdk
Audio Duration: 59 seconds
 [Image] image_bg_1: https://drive.google.com/file/d/1KwCp3hnAKyZv4fT0-irZoKQmlfQkOEea/view?usp=drivesdk
 [Text] title_text_1: Automate Your Content:
 [Text] subtitle_text_1: From Idea to LIVE Article in Minutes
 [Text] bullet_point_1_0: Unlock unparalleled speed and efficiency in your content workflow.
 [Text] bullet_point_1_1: Transform how you create, publish, and scale your content efforts.

Scene 2 (Slide 2)
Audio URL: https://drive.google.com/file/d/1v2IFExmN8fkWl3Jdd3jVL2YlIKNNKAA-/view?usp=drivesdk
Audio Duration: 94 seconds
 [Image] image_bg_2: https://drive.google.com/file/d/1wL6aYHY_0MXvrDGOL7h_jvTbKiMnrhHQ/view?usp=drivesdk
 [Text] title_text_2: The Content Creation Conundrum:
 [Text] subtitle_text_2: Why Ma

In [2]:
import json
import os
import re
import gdown
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Animation:
    type: str
    startTime_sec: Optional[float] = None
    duration_sec: Optional[float] = None
    start_zoom: Optional[float] = None
    end_zoom: Optional[float] = None
    direction: Optional[str] = None

@dataclass
class Position:
    x: int
    y: int
    anchor: str

@dataclass
class Layer:
    layer_id: str
    type: str
    content: Optional[str] = None
    url: Optional[str] = None
    font: Optional[str] = None
    size: Optional[int] = None
    color: Optional[str] = None
    position: Optional[Position] = None
    animation: Optional[Animation] = None

@dataclass
class Scene:
    scene_id: int
    slide_id: int
    audioUrl: Optional[str]
    audioDuration_sec: float
    layers: List[Layer] = field(default_factory=list)

@dataclass
class VideoMetadata:
    title: str
    resolution: str
    fps: int

@dataclass
class VideoProject:
    metadata: VideoMetadata
    scenes: List[Scene]

def load_project_from_json(json_path: str) -> VideoProject:
    with open(json_path, "r") as f:
        data = json.load(f)

    metadata = VideoMetadata(
        title=data["videoMetadata"]["title"],
        resolution=data["videoMetadata"]["resolution"],
        fps=data["videoMetadata"]["fps"]
    )

    scenes = []
    for s in data["scenes"]:
        layers = []
        for l in s["layers"]:
            position = Position(**l["position"]) if "position" in l else None
            animation = Animation(**l["animation"]) if "animation" in l else None
            layers.append(Layer(
                layer_id=l["layer_id"],
                type=l["type"],
                content=l.get("content"),
                url=l.get("url"),
                font=l.get("font"),
                size=l.get("size"),
                color=l.get("color"),
                position=position,
                animation=animation
            ))
        scenes.append(Scene(
            scene_id=s["scene_id"],
            slide_id=s["slide_id"],
            audioUrl=s.get("audioUrl"),
            audioDuration_sec=s["audioDuration_sec"],
            layers=layers
        ))

    return VideoProject(metadata=metadata, scenes=scenes)

def extract_file_id(drive_url: str) -> Optional[str]:
    """Extract Google Drive file ID from shared URL."""
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", drive_url)
    return match.group(1) if match else None

def download_assets(project: VideoProject, output_dir: str = "downloads"):
    """Download all Google Drive assets (audio + images) for each scene."""
    os.makedirs(output_dir, exist_ok=True)

    for scene in project.scenes:
        scene_folder = os.path.join(output_dir, f"scene_{scene.scene_id}")
        os.makedirs(scene_folder, exist_ok=True)

        # Download audio file if present
        if scene.audioUrl:
            file_id = extract_file_id(scene.audioUrl)
            if file_id:
                audio_output = os.path.join(scene_folder, f"audio_scene_{scene.scene_id}.mp3")
                gdown.download(f"https://drive.google.com/uc?export=download&id={file_id}", audio_output, quiet=False)

        # Download image layers if present
        for layer in scene.layers:
            if layer.type == "image" and layer.url:
                file_id = extract_file_id(layer.url)
                if file_id:
                    img_output = os.path.join(scene_folder, f"{layer.layer_id}.png")
                    gdown.download(f"https://drive.google.com/uc?export=download&id={file_id}", img_output, quiet=False)

# Example usage
project = load_project_from_json("scene_composition_agent_output.json")
print(f"Loaded {len(project.scenes)} scenes")

download_assets(project)  # This will create folders and download assets


Loaded 8 scenes


Downloading...
From: https://drive.google.com/uc?export=download&id=16yvluYop5Q3rqb5EXa8rK_1BnsE_cMFO
To: c:\Minh\workplace\Internship\downloads\scene_1\audio_scene_1.mp3
100%|██████████| 525k/525k [00:00<00:00, 7.48MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=1KwCp3hnAKyZv4fT0-irZoKQmlfQkOEea
To: c:\Minh\workplace\Internship\downloads\scene_1\image_bg_1.png
100%|██████████| 1.22M/1.22M [00:00<00:00, 4.91MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=1v2IFExmN8fkWl3Jdd3jVL2YlIKNNKAA-
To: c:\Minh\workplace\Internship\downloads\scene_2\audio_scene_2.mp3
100%|██████████| 800k/800k [00:00<00:00, 3.71MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=1wL6aYHY_0MXvrDGOL7h_jvTbKiMnrhHQ
To: c:\Minh\workplace\Internship\downloads\scene_2\image_bg_2.png
100%|██████████| 1.76M/1.76M [00:00<00:00, 6.21MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=1UGUgQXYGNRvrgdxNny_uE3rXnf5jg5vM
To: c:\Minh\